# Neptune Motion: Qwen-Image 2.1 Diffusion Pipeline on Kaggle

This notebook executes the complete end-to-end text-to-image pipeline for **Qwen-Image 2.1** using ComfyUI with the Neptune Motion architecture:
- **Diffusion Backbone**: `qwen-image-2.1-UC-Q4_K_M.gguf` (Quantized Turbo DiT ~3.8 GB)
- **Vision-Language Text Encoder**: `qwen3vl_8b_int8_convrot.safetensors` (Qwen3-VL 8B INT8 ~8.9 GB)
- **Autoencoder**: `qwen_image_2.1_vae_bf16.safetensors` (Native Bfloat16 Autoencoder ~644 MB)

### Kaggle Settings:
1. **Accelerator**: Select **GPU (T4 x2 or P100)** or **TPU** (if available) in the right-hand panel.
2. **Internet**: Toggle **Internet ON** in Notebook Settings (required to clone repo and download models).


In [ ]:
# Step 1: Environment & Hardware Discovery
import os, sys, subprocess, shutil, time, json, glob, importlib.util

print(f"Python Version: {sys.version}")

# Check NVIDIA GPU safely (nvidia-smi is absent on TPU/CPU VMs)
has_gpu = False
if shutil.which("nvidia-smi"):
    has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"NVIDIA GPU Available: {has_gpu}")
if has_gpu:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])

# Check TPU devices (/dev/accel* or /dev/vfio/* or PJRT env)
has_tpu = (
    bool(glob.glob("/dev/accel[0-9]*"))
    or bool(glob.glob("/dev/vfio/[0-9]*"))
    or "TPU" in os.environ.get("PJRT_DEVICE", "").upper()
    or "TPU_NAME" in os.environ
    or "KAGGLE_TPU_NAME" in os.environ
)
print(f"Google Cloud TPU Hardware Available: {has_tpu}")
if has_tpu:
    os.environ["TPU_SKIP_MDS_QUERY"] = "1"
    os.environ["PJRT_DEVICE"] = "TPU"
    os.environ.setdefault("TPU_ACCELERATOR_TYPE", "v5e-8")
    os.environ.setdefault("TPU_CHIPS_PER_HOST_BOUNDS", "2,2,1")
    os.environ.setdefault("TPU_HOST_BOUNDS", "1,1,1")
    os.environ.setdefault("TPU_WORKER_ID", "0")
    os.environ.setdefault("TPU_RUNTIME_METRICS_PORTS", "8431,8432,8433,8434")
    os.environ.setdefault("TPU_WORKER_HOSTNAMES", "localhost")

print("Environment initialized.")


In [ ]:
# Step 2: Clone ComfyUI and Install Dependencies
COMFY_DIR = "/kaggle/working/ComfyUI"
if not os.path.exists(COMFY_DIR):
    print("Cloning ComfyUI (feat/tpu-multi-core-support)...")
    subprocess.run([
        "git", "clone", "--depth", "1", "-b", "feat/tpu-multi-core-support",
        "https://github.com/vuthanhtrung2010/ComfyUI.git", COMFY_DIR
    ], check=True)
else:
    print(f"ComfyUI already cloned at {COMFY_DIR}")

# Install torch_tpu on TPU if missing, and install ComfyUI dependencies without overwriting preinstalled torch
if has_tpu and importlib.util.find_spec("torch_tpu") is None:
    print("Installing PyTorch CPU + torch_tpu for TPU...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cpu"
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "torch_tpu"
    ], check=True)

print("Installing dependencies (takes ~30-45s)...")
skip_torch = has_tpu or (importlib.util.find_spec("torch") is not None)
with open(f"{COMFY_DIR}/requirements.txt", "r") as f:
    reqs = [
        l.strip() for l in f
        if l.strip() and not l.strip().startswith("#")
        and not (skip_torch and l.strip().lower().startswith(("torch", "torchvision", "torchaudio")))
    ]
req_tmp = "/tmp/comfy_reqs.txt"
with open(req_tmp, "w") as f:
    f.write("\n".join(reqs) + "\n")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", req_tmp, "gguf", "torchsde"
], check=True)

# Install ComfyUI-GGUF custom node
gguf_dir = os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI-GGUF")
if not os.path.exists(gguf_dir):
    print("Cloning ComfyUI-GGUF node...")
    subprocess.run(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", gguf_dir], check=True)

# Apply Neptune Motion patch for Qwen-Image 2.1 and TPU weight caching
patch_script = os.path.join(COMFY_DIR, "tools", "patch_comfyui_gguf.py")
if os.path.exists(patch_script):
    subprocess.run([sys.executable, patch_script, gguf_dir], check=True)
print("ComfyUI-GGUF patched successfully for Qwen-Image 2.1 and weight caching!")


In [ ]:
# Step 3: Download Qwen-Image 2.1 Model Checkpoints
def _resolve_hf_token():
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = None
    return tok.strip() if isinstance(tok, str) and tok.strip() else None

hf_token = _resolve_hf_token()
print(f"HuggingFace auth mode: {'HF_TOKEN active' if hf_token else 'anonymous (set HF_TOKEN env or Kaggle Secret for higher speed/quota)'}")

def download_checkpoint(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 100 * 1024 * 1024:
        print(f"[READY] Already cached: {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.1f} MB)")
        return
    print(f"[DOWNLOADING] {os.path.basename(dest)}...")
    t0 = time.time()
    cmd = ["curl", "-L", "-C", "-", "--retry", "5", "-o", dest]
    if hf_token:
        cmd += ["-H", f"Authorization: Bearer {hf_token}"]
    cmd.append(url)
    subprocess.run(cmd, check=True)
    dt = max(time.time() - t0, 1e-3)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"[COMPLETE] {os.path.basename(dest)} ({size_mb:.1f} MB in {dt:.1f}s, {size_mb / dt:.1f} MB/s)")

models = [
    ("models/unet/qwen-image-2.1-UC-Q4_K_M.gguf",
     "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/qwen-image-2.1-turbo-Q4_K_M.gguf"),
    ("models/clip/qwen3vl_8b_int8_convrot.safetensors",
     "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors"),
    ("models/vae/qwen_image_2.1_vae_bf16.safetensors",
     "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors")
]

for rel_path, url in models:
    dest = os.path.join(COMFY_DIR, rel_path)
    download_checkpoint(url, dest)

print("All 3 Qwen-Image 2.1 model checkpoints downloaded and verified!")


In [ ]:
# Step 4: Launch ComfyUI Server in Background
import urllib.request
os.chdir(COMFY_DIR)

# Terminate existing server if re-running cell
subprocess.run(["pkill", "-f", "main.py"])
time.sleep(1)

# Configure launch flags based on detected hardware
flags = [sys.executable, "main.py", "--port", "8188", "--listen", "127.0.0.1"]
if has_tpu:
    print("Backend: Google Cloud TPU (--tpu-device 0)")
    flags += ["--tpu-device", "0"]
elif has_gpu:
    print("Backend: NVIDIA GPU (--lowvram)")
    flags += ["--lowvram"]
else:
    print("Backend: CPU fallback")
    flags += ["--cpu"]

server_log_path = "/kaggle/working/comfyui_server.log"
server_log = open(server_log_path, "w")
server_proc = subprocess.Popen(flags, stdout=server_log, stderr=subprocess.STDOUT, cwd=COMFY_DIR)
print(f"Server process launched with PID {server_proc.pid}. Waiting for server readiness...")

server_ready = False
for i in range(90):
    if server_proc.poll() is not None:
        print(f"ERROR: ComfyUI server exited early with code {server_proc.returncode}")
        break
    time.sleep(1)
    try:
        req = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if req.status == 200:
            stats = json.loads(req.read().decode())
            devices = [d["name"] for d in stats.get("devices", [])]
            print(f"SUCCESS: ComfyUI Server is ONLINE!")
            print(f"Detected Devices: {devices}")
            server_ready = True
            break
    except Exception:
        pass

if not server_ready:
    print(f"Server failed to become ready. Tail of {server_log_path}:")
    if os.path.exists(server_log_path):
        with open(server_log_path, "r", errors="replace") as f:
            print("".join(f.readlines()[-40:]))


In [ ]:
# Step 5: Execute Neptune Motion Qwen-Image 2.1 Workflow
import urllib.request, urllib.error
PROMPT_TEXT = "An artistic educational diagram explaining Shannon information theory, entropy formula, and communication channel, detailed whiteboard illustration, sharp clean lines"
NEGATIVE_TEXT = "blurry, low quality, distorted, artifacts, photo frame, watermark"

workflow = {
    "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": "qwen-image-2.1-UC-Q4_K_M.gguf"}},
    "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image", "device": "default"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}},
    "4": {"class_type": "TextEncodeQwenImage21", "inputs": {
        "clip": ["2", 0],
        "prompt": PROMPT_TEXT,
        "negative_prompt": NEGATIVE_TEXT,
        "resolution": 1024
    }},
    "5": {"class_type": "EmptyLatentImage", "inputs": {"width": 1024, "height": 640, "batch_size": 1}},
    "6": {"class_type": "KSampler", "inputs": {
        "model": ["1", 0],
        "seed": 42,
        "steps": 8,
        "cfg": 1.0,
        "sampler_name": "euler",
        "scheduler": "simple",
        "positive": ["4", 0],
        "negative": ["4", 1],
        "latent_image": ["5", 0],
        "denoise": 1.0
    }},
    "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}},
    "8": {"class_type": "SaveImage", "inputs": {"images": ["7", 0], "filename_prefix": "explainer"}}
}

print(f"Queuing Qwen-Image 2.1 Workflow...")
print(f"Prompt: {PROMPT_TEXT}")

data = json.dumps({"prompt": workflow, "client_id": "kaggle_manual_run"}).encode("utf-8")
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data, headers={"Content-Type": "application/json"})
resp = urllib.request.urlopen(req)
prompt_res = json.loads(resp.read().decode())
prompt_id = prompt_res["prompt_id"]
print(f"Prompt successfully queued! Task ID: {prompt_id}")

# Monitor execution
start_time = time.time()
completed = False
while time.time() - start_time < 900:
    time.sleep(3)
    try:
        h_req = urllib.request.urlopen("http://127.0.0.1:8188/history")
        history = json.loads(h_req.read().decode())
        if prompt_id in history:
            entry = history[prompt_id]
            status = entry.get("status", {})
            if status.get("completed", False):
                elapsed = time.time() - start_time
                print(f"\n[SUCCESS] Image Generation Completed in {elapsed:.1f} seconds!")
                completed = True
                break
            elif status.get("status_str") == "error":
                print(f"\n[ERROR] Workflow failed: {status}")
                break
    except Exception:
        pass
    
    elapsed = int(time.time() - start_time)
    print(f"\rDenoising and decoding in progress... {elapsed}s elapsed", end="", flush=True)


In [ ]:
# Step 6: Display and Save Generated Image Artifact
from IPython.display import Image, display

output_dir = os.path.join(COMFY_DIR, "output")
found_images = []
if os.path.exists(output_dir):
    for fname in os.listdir(output_dir):
        if fname.endswith((".png", ".jpg", ".webp")):
            src_path = os.path.join(output_dir, fname)
            dest_path = os.path.join("/kaggle/working", fname)
            shutil.copy(src_path, dest_path)
            found_images.append(dest_path)

if found_images:
    print(f"\n=== Generated Images ({len(found_images)}) ===")
    for p in found_images:
        print(f"File saved to: {p} ({os.path.getsize(p) / 1024:.1f} KB)")
        display(Image(filename=p))
else:
    print("No images found. Check /kaggle/working/comfyui_server.log")
